In [28]:
from database.database import DataBase
import pandas as pd

### Database for SQL Questions Part

In [29]:
db_path = "database/sql_questions.db"

### Create connection from the Database

In [30]:
conn = DataBase(path=db_path).conn

In [31]:
class SQLite3Connection:
    def __init__(
        self,
        db_path: str = "database/sql_questions.db"
    ):
        self.db_path = db_path
        self.conn = DataBase(path=self.db_path).conn

    @staticmethod
    def parse_query(
        query_path:str=None
    )-> str:
        with open(query_path, "r") as query:
            return query.read()

    def qr_to_df(
        self,
        query: str
    ) -> pd.DataFrame:
        cursor = self.conn.execute(query)
        data = cursor.fetchall()
        header = [col[0] for col in cursor.description]
        return pd.DataFrame(data=data, columns=header)

In [32]:
data = SQLite3Connection()

### Testing the datatype among the database

### Q1: Question:
<span style="font-family: Arial;font-size: 13px;color: #FFFF;">Write an SQL query to classify buyers of each Level-1 category in the last 30 days into: </span><span style="font-family: Arial;font-size: 13px;color: #FF0000;">new-to-category</span><span style="font-family: Arial;font-size: 13px;color: #FFFF;"> and </span><span style="font-family: Arial;font-size: 13px;color: #0000FF;">new-to-platform.</span><span style="font-family: Arial;font-size: 13px;color: #FFFF;"> Also identify new-new users, meaning users whose first platform order and first category order happened in the same order.</span>

Display-Result:
- category_name_level_1_id
- buyer_type
- users




### Answer-Q1:
```sql
/* 
issues-date: 2026-09-26 22:30:00
commented by: Pham Ngoc Huy
data from public.dwd_order
- order_id: being duplicated from ORD008: 
    # fix: 8,9,10
- user_id: user_id in dim_user sample only 1 -> 6 but data appear 6->8
    # fix: 1 -> 5

Note: all the [id key] -- is being wrong datatype at the beginning
- schema tells *_id: integer - but datasample: varchar
    # fix: remove all char "[A-z]"

database: PostgreSQL
*/

-- Question 1: 
/*
Descrtions:
Write an SQL query: 
    to classify buyers of each Level-1 category in the last 30 days into: 
        - new-to-category 
        - new-to-platform
    Also identify new-new users, meaning users whose first platform order and first category order happened in the same order.

Display results:
    - category_name_level_1_id
    - buyer_type
    - users
*/
-- considered: if order happens, it will track -- no filtered in order 'cancel' or not.
with platform_firsts as (
-- objective: find very-first order time-stamp for each user accross the entire platform
select
user_id,
min(created_time) as first_platform_time
from public.dwd_order
where 1=1
group by
    user_id
),
category_firsts as (
-- objective: find very-first order time-stamp for each user within each specific category_name_level_1_id
select
user_id,
category_name_level_1_id,
min(created_time) as first_category_time
from dwd_order
where 1=1
group by
    user_id,
    category_name_level_1_id
),
classify_buyers as (
select
c.category_name_level_1_id,
c.user_id,
case 
    -- happen in the same order --> new-new
    when c.first_category_time = p.first_platform_time then 'new-new'
    -- joined the platform in the last 30 days, but this category wasn't their first order
    when p.first_platform_time >= (select max(created_time) - interval '30 days' FROM public.dwd_order) THEN 'new-to-platform'
-- Order-platform user buying in this category for the first time
else 'new-to-category'
end as buyer_type
from category_firsts c
join platform_firsts p
    on c.user_id = p.user_id
    -- filtered: last 30 days from 'buyers of each Level-1 category'
    and c.first_category_time >= (select max(created_time) - interval '30 days' from public.dwd_order)
)
select 
category_name_level_1_id,
buyer_type,
count(distinct user_id) as users
from classify_buyers
group by
    category_name_level_1_id,
    buyer_type
order by
    category_name_level_1_id,
    buyer_type;
```

### Q2: Question 
<span style="font-family: Arial;font-size: 13px;color: #FFFF;">Write a SQL query to calculate 30-day </span><span style="font-weight: bold;font-family: Arial;font-size: 13px;color: #FFFF;">conversion by user registration monthly cohort</span><span style="font-family: Arial;font-size: 13px;color: #FFFF;">. A user is considered converted if they place at least one completed order within 30 days after registration.</span>

Display-Result:
- reg_month
- new_users
- buyers_30d
- conversion_30d


### Answer-Q2:
```sql
/*
Question 2:
Write a SQL query to calculate 30-day conversion by user registration monthly cohort. 
A user is considered converted if 
    they place at least one completed order within 30 days after registration.

Display results:
    - reg_month
    - new_users
    - buyers_30d
    - conversion_30d
*/
with converted_users as (
select distinct
u.user_id
from public.dim_users u
join public.dwd_order o
    on u.user_id = o.user_id
        and upper(o.order_status) = 'COMPLETED'
        and o.created_time >= u.created_time
        and o.created_time <= u.created_time + interval '30 days'
)
select 
to_char(u.created_time, 'YYYY-MM') as reg_month,
count(distinct u.user_id) as new_users,
count(distinct c.user_id) as buyers_30d,
round(
    count(distinct c.user_id) * 100.00 / nullif(count(distinct u.user_id), 0),
    2
) as conversion_30d
from public.dim_users u
left join converted_users c
    on u.user_id = c.user_id
group by
    to_char(u.created_time, 'YYYY-MM')
order by
    reg_month
;
```

### Q3: Question
Write a SQL query to identify users whose second purchase happened faster than 80% of other users. Use only completed orders and calculate the time difference between the first and second purchase.

Display-Result:
- user_id
- first_order_time
- second_order_time
- minutes_between_orders

### Answer-Q3:
```sql
/*
Question 3:
Write a SQL query to identify users whose second purchase happened faster than 80% of other users. 
Use only completed orders and calculate the time difference between the first and second purchase.

Display results:
- user_id
- first_order_time
- second_order_time
- minutes_between_orders
*/
with user_orders as (
-- Get: "Completed" Orders and the previous order-time
select
user_id,
lag(created_time) over (partition by user_id order by created_time) as first_order_time,
created_time as second_order_time,
row_number() over (partition by user_id order by created_time) as rn
from public.dwd_order 
where 1=1
    and upper(order_status) = 'COMPLETED'
),
time_calculated as (
-- keep the second order only (rn=2) and calculat the minutes
select
user_id,
first_order_time,
second_order_time,
extract(epoch from (second_order_time - first_order_time)) / 60 as minutes_between_orders
from user_orders
where 1=1
    and rn = 2
),
percentile_ranked as (
select
user_id,
first_order_time,
second_order_time,
minutes_between_orders,
percent_rank() over (order by minutes_between_orders) as percent_rank
from time_calculated
)
-- Output: 20% users has the fastest
select 
user_id,
first_order_time,
second_order_time,
round(minutes_between_orders,2) as minutes_between_orders
from percentile_ranked
where 1=1
    and percent_rank <= 0.20
order by
    minutes_between_orders asc;
```

### Q4: Question 
Write an SQL query to split users into two statistically comparable experiment groups (A/B) using a deterministic method. Then write SQL to compare historical metrics (orders, GMV, avg basket size) between the two groups to validate balance.

Display-Result:
- experiment_group
- users
- avg_orders_90d
- avg_gmv_90d
- avg_basket_size

### Answer-Q4:
```sql
/*
Question 4:
Write an SQL query to split users into 
    two statistically comparable experiment groups (A/B) using a deterministic method
Then write SQL to compare historical metrics (orders, GMV, avg basket size) 
    between the two groups to validate balance.

Display results:
- experiment_group
- users
- avg_orders_90d
- avg_gmv_90d
- avg_basket_size
*/
with user_groups as (
select
user_id,
case
    when mod(user_id,2) = 0 then 'Group A'
else 'Group B'
end as experiment_group
from public.dim_users
),
user_metrics as (
-- calculate the 90-day performance for every user (including thoses with 0 orders)
select
ug.user_id,
ug.experiment_group,
count(o.order_id) as total_orders,
coalesce(sum(o.gmv), 0) as total_gmv
from user_groups ug
left join public.dwd_order o
    on ug.user_id = o.user_id
    and upper(o.order_status) = 'COMPLETED'
    and o.created_time >= (select max(created_time) - interval '90 days' from public.dwd_order)
group by
    ug.user_id,
    ug.experiment_group
)
select
experiment_group,
count(user_id) as users,
round(avg(total_orders),2) as avg_orders_90d,
round(avg(total_gmv),2) as avg_gmv_90d,
round(sum(total_gmv) / nullif(sum(total_orders),0),2) as avg_basket_size
from user_metrics
group by
    experiment_group
order by
    experiment_group

```